In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from matplotlib.pyplot import imshow
from PIL import Image
import os
from pathlib import Path

# Ignore warnings
import warnings
warnings.filterwarnings("ignore")

torch.__version__

'2.8.0'

In [2]:
data_dir = Path("/Users/akshay/Work/data")

# Dataset and DataLoader Class

PyTorch provides two data primitives: `torch.utils.data.DataLoader` and `torch.utils.data.Dataset` that allow to use pre-loaded datasets as well as your own data.

## Dataset class and loading dataset

`Dataset` stores the samples and their corresponding labels.

In [3]:
from torch.utils.data import Dataset, DataLoader

# Set random seed for reproducibility
# This ensures that random operations (like data generation) produce the same results
# across different runs, which is important for debugging and consistent experiments
torch.manual_seed(1234)

## Creating a custom dataset

`torch.utils.data.Dataset` is an abstract class representing a dataset. Custom dataset should inherit Dataset and override,
1. \_\_init\_\_(): The \_\_init\_\_ function is run once when instantiating the Dataset object. We initialize the directory containing the images, the annotations file, and both transforms
2. \_\_len\_\_(): The \_\_len\_\_ function returns the number of samples in our dataset.
3. \_\_getitem\_\_(): The \_\_getitem\_\_ function loads and returns a sample from the dataset at the given index `index`.

In [4]:
#Class to load sample dummy dataset
class toyDataset(Dataset):
    """
        input : random integers of size (length, 128) with values between 0 to max
        label : 1 if sum of row is even; else 0
    """
    #Constructor with default values
    #Where to load data from, whether to apply any transformation on data, etc
    def __init__(self, length=10, max=1000, transform=None, target_transform=None):
        self.len = length
        # x :: input features
        # Usually we load data from files. Here, for simplicity, we are generating random data
        self.x = torch.randint(0, max,(length, 128), dtype=torch.float32)
        
        # y :: target labels
        # Usually this can be part of the files containing input features, or in separate files
        self.y = torch.Tensor(list(map(lambda x : int(sum(x) % 2 == 0), self.x)))
        
        #Whether data features need to transformed (like, normalization, etc)
        self.transform = transform
        
        #Whether data labels need to transformed (like, generating one-hot vectors, etc)
        self.target_transform = target_transform
        
    #Method overriding to return the total number of instances 
    def __len__(self):
        return self.len
    
    #Method overriding to return data samples
    def __getitem__(self, index):
        if self.transform:
            self.x[index] = self.transform(self.x[index])
        if self.target_transform:
            self.y[index] = self.transform(self.y[index])
        sample = self.x[index], self.y[index]
        return sample
    
    ## Optional: One can define custom functions for data loading, manipulation, etc.
    ## mostly for image datasets, one may define a function to read image files
    ## this will avoid loading all images at once in memory in __init__()

In [5]:
#Creating instance of toyDataset and accessing example instances
data = toyDataset(length=1000)
print("Total number of samples in dataset = ", len(data))

Total number of samples in dataset =  1000


In [6]:
# Accessing a specific sample from the dataset
# The __getitem__ method is called when using indexing
data[999]

(tensor([ 46., 156., 935., 330., 449., 867., 504., 213.,  66.,  75.,  87., 260.,
         955., 182., 144., 962., 606., 108., 193., 267., 425., 595.,  22.,  88.,
         204., 359., 635., 114., 851., 993., 527., 383.,  37., 324., 654., 159.,
         108., 569., 702., 293., 785., 663., 750., 736., 858., 659., 555., 593.,
         345., 900., 190., 198., 184., 421.,   1.,  60., 785., 986., 320.,  19.,
         413., 647., 601., 708., 193., 728., 196.,  18., 902., 437., 851., 149.,
         610., 900., 604., 260., 279., 436., 790., 974., 967., 237.,  85., 483.,
         838., 626., 747., 235., 322., 277., 987., 575., 567., 707., 504., 697.,
         814., 343., 613., 953., 841., 954., 693., 523., 539., 242., 253.,  72.,
          54., 858., 676., 672., 446.,  98.,  63., 639., 311., 327., 567., 646.,
         534., 286., 872., 531., 476., 304., 927., 379.]),
 tensor(0.))

### Transform
Most of the time, we need to do some type of transformation in the dataset, like normalizing the data, setting the image size, etc. Thus, there is need to write some pre-processing code. <br>
It is ideal to implement them as class rather than functions.

#### Custom transformations

In [7]:
class transform_my_data(object):
    def __init__(self, transformation_params):
        """
            Constructor
        """
        self.tranformation_params = transformation_params
    
    def __call__(self, x):
        """
            Executor:
            Necessary tranformation
            to each instance of data.
            
        """
        x *= self.tranformation_params
        
        return x       

In [8]:
class Normalize(object):
    """
    Custom normalization transform for 1D tensors.
    
    Note: torchvision.transforms.Normalize is designed specifically for image tensors
    (with shape [C, H, W] or [H, W]). For 1D tensors or non-image data, we need to 
    create a custom normalization transform like this one.
    """
    def __init__(self, mean, std):
        """
            Constructor
            
            Args:
                mean: Mean value for normalization
                std: Standard deviation for normalization
        """
        self.mean = mean
        self.std = std
    
    def __call__(self, tensor):
        """
            Executor:
            Normalize the tensor with mean and standard deviation.
            
            Formula: (tensor - mean) / std
        """
        tensor = (tensor - self.mean) / self.std
        
        return tensor

Creating instance of transform and using transform parameter from our dataset's constructor, we can initialize transformation in our dataset.

In [9]:
transform = transform_my_data(1.2)

In [10]:
mean = 499.0
std = 288.8

normalise = Normalize(mean, std)

In [11]:
transformed_dataset = toyDataset(transform=transform)
normalised_dataset = toyDataset(transform=normalise)

In [12]:
print(data[9])
print(transformed_dataset[9])
print(normalised_dataset[9])


(tensor([765., 316., 821., 900., 213., 713., 142.,  67., 994., 931., 614., 808.,
        277., 242., 403., 432.,  22., 850., 624.,  34., 475., 159., 115., 728.,
        938., 662., 850., 291., 779.,  72., 915., 499., 895., 659., 504., 355.,
        412., 121., 648., 460., 197., 762., 370., 446., 911., 345., 822., 250.,
        508., 165., 388., 482., 522., 763.,  33., 341., 551., 925., 637., 809.,
        709., 543., 925., 403., 535., 911., 881., 232., 445., 809., 369., 992.,
        814., 443., 341., 876., 475., 364., 471., 444., 898., 268., 266., 861.,
        335., 430., 868., 538., 582., 281., 175., 126., 482., 234., 315., 731.,
        785., 940., 737.,  76., 249., 581., 602., 704., 439., 603., 523., 812.,
         86., 165., 868., 648., 448., 466., 290., 862., 733., 948., 914., 705.,
        710., 260., 349., 675.,   3., 570., 916.,  70.]), tensor(0.))
(tensor([ 846.0001,  758.4000,  786.0001, 1077.6001,  643.2000,  822.0001,
         488.4000,   39.6000, 1081.2001,  936.0001,  4

### Composing multiple transform

In [13]:
from torchvision import transforms

In [14]:
data_transform = transforms.Compose([transform, normalise])
data_transform

Compose(
)

The `Compose` object will perform each transform sequentially (in the order specified).

In [15]:
dataset = toyDataset(transform=data_transform)

In [16]:
print(dataset[5])

(tensor([-0.5727,  0.3165,  2.2777, -0.9799, -1.2126, -0.5395,  1.1434,  2.3608,
         0.2417, -1.0630,  0.2832,  0.5907,  0.6572,  1.0104,  1.9328,  0.0506,
         0.2957, -0.8677, -1.5201,  0.0589,  2.2819, -1.1877, -0.0450,  0.0422,
         0.8774, -0.8802, -0.9924, -0.9508,  1.6420,  1.3137,  1.4882,  0.2583,
        -0.7514,  1.5838, -1.0713, -1.4162, -1.7278,  1.5547,  2.3566,  0.9813,
        -1.2957,  0.0422,  1.8456,  0.2500, -0.6350,  2.0367,  0.9481,  1.2680,
        -1.3123,  1.2431, -0.2029, -1.0713, -1.1544, -1.2084, -1.4619,  1.1724,
        -0.8345, -1.5575,  1.3220, -1.3539,  1.5090,  0.0381,  0.7943,  0.1170,
         1.4384,  0.2292,  2.2029, -0.5021, -0.6350, -1.3040, -0.4231,  1.7168,
         1.7666,  2.2777,  1.4384, -1.2666,  0.5741,  1.3179, -0.0866,  2.1447,
         2.3109, -0.0658,  0.8774, -0.8262, -0.0367, -0.6226,  0.5201, -1.6198,
        -1.1918,  2.3151, -0.4481, -0.9716, -1.0963, -0.9134,  1.0976, -0.7306,
        -0.0824, -1.3414,  1.9952,  1.4

### Splitting dataset into train, validation, and test sets

The `split_dataset` function will use `torch.utils.data.random_split` to split the dataset into train, validation, and test split.

In [17]:
from torch.utils.data import random_split

In [18]:
def split_dataset(dataset, val_frac=0.1, test_frac=0.1):
    """
        Splits the dataset into train, validation, and test sets.
        
        Args:
            dataset: The dataset to be split.
            val_frac: Fraction of data to be used for validation.
            test_frac: Fraction of data to be used for testing.
    """
    total_size = len(dataset)
    test_size = int(total_size * test_frac)
    val_size = int(total_size * val_frac)
    train_size = total_size - val_size - test_size
    
    train_set, val_set, test_set = random_split(dataset, [train_size, val_size, test_size])
    
    return train_set, val_set, test_set

In [19]:
dataset = toyDataset(length=10000, transform=data_transform)
train_set, val_set, test_set = split_dataset(dataset, val_frac=0.15, test_frac=0.15)
print(f"Train set size: {len(train_set)}")
print(f"Validation set size: {len(val_set)}")
print(f"Test set size: {len(test_set)}")

Train set size: 7000
Validation set size: 1500
Test set size: 1500


### Dealing with real dataset

So far,
1. dataset was not real and was small, therefore we initialised at __init__(), which must not be done for real datasets, as it will load the entire dataset at once, consuming large memory.
2. we have iterated through the dataset using for loop, where we miss various features like, batching, shuffling, load the data in multiprocessing environment. Hence we will use dataloader (iterator).

#### Iterating over dataset using DataLoader
The `Dataset` retrieves our dataset’s `features` and `labels` one sample at a time. While training a model, we typically want to pass samples in “minibatches”, reshuffle the data at every epoch to reduce model overfitting, and use Python’s multiprocessing to speed up data retrieval. `torch.utils.data.DataLoader` wraps an iterable around the `Dataset` to enable easy access to the samples.

In [20]:
# DataLoader parameters:
# - batch_size: Number of samples per batch
# - shuffle: Whether to shuffle the data (True for training, False for validation/test)
train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
val_loader = DataLoader(val_set, batch_size=64, shuffle=False)
test_loader = DataLoader(test_set, batch_size=64, shuffle=False)

In [21]:
for batch_idx, (data, target) in enumerate(train_loader):
    print(f"Batch {batch_idx+1}:")
    print(f"Data: {data}")
    print(f"Target: {target}")
    break

Batch 1:
Data: tensor([[-0.9799, -0.9758,  0.1212,  ...,  1.7791,  2.2029,  0.8234],
        [ 1.2389,  1.9453,  1.1226,  ...,  0.4328, -0.9550,  1.6918],
        [ 1.7126,  0.2084, -0.3068,  ...,  1.5755,  2.2195, -1.4536],
        ...,
        [ 2.2486,  2.0741, -0.7223,  ..., -0.4855, -0.9633, -0.1406],
        [-0.7805,  1.6420,  1.5672,  ...,  1.2181, -1.5699,  0.9024],
        [ 2.3234,  1.5547,  1.3428,  ...,  2.2569,  2.3566, -0.3982]])
Target: tensor([1., 0., 1., 1., 0., 1., 1., 1., 0., 0., 1., 1., 1., 0., 1., 1., 0., 0.,
        1., 0., 1., 1., 0., 1., 0., 0., 1., 0., 0., 1., 0., 0., 0., 0., 1., 0.,
        0., 0., 0., 0., 1., 0., 1., 0., 1., 0., 1., 1., 1., 1., 0., 0., 0., 0.,
        1., 1., 0., 0., 0., 1., 1., 0., 0., 0.])


In [22]:
import time
from tqdm import tqdm

n_epochs = 2
train_batches = len(train_loader)
val_batches = len(val_loader)

for epoch in range(n_epochs):
    print(f"Epoch {epoch+1}/{n_epochs}")
    print("-" * 20)
    print("Training:")
    for batch_idx, (data, target) in tqdm(enumerate(train_loader), desc=f"Epoch {epoch}", unit="batch", total=train_batches):
        # Here you would typically perform your training step
        pass
    
    print("Validation:")
    for batch_idx, (data, target) in tqdm(enumerate(val_loader), desc=f"Epoch {epoch}", unit="batch", total=val_batches):
        # Here you would typically perform your validation step
        pass

# Testing loop
eval_batches = len(test_loader)
print("Testing:")
for batch_idx, (data, target) in tqdm(enumerate(test_loader), desc=f"Evaluation", unit="batch", total=eval_batches):
    # Here you would typically perform your testing step
    pass

Epoch 1/2
--------------------
Training:


Epoch 0: 100%|██████████| 110/110 [00:00<00:00, 2145.91batch/s]


Validation:


Epoch 0: 100%|██████████| 24/24 [00:00<00:00, 2471.66batch/s]


Epoch 2/2
--------------------
Training:


Epoch 1: 100%|██████████| 110/110 [00:00<00:00, 2753.32batch/s]


Validation:


Epoch 1: 100%|██████████| 24/24 [00:00<00:00, 2587.88batch/s]


Testing:


Evaluation: 100%|██████████| 24/24 [00:00<00:00, 2523.33batch/s]




### Working with Text dataset

Text data presents unique challenges compared to structured data:
- **Variable length sequences**: Different texts have different lengths
- **Tokenization**: Text needs to be converted to numerical tokens
- **Padding/Truncation**: Sequences need to be standardized for batching
- **Special tokens**: End-of-text, padding, and other special markers may be needed

In this section, we'll create a custom dataset for text data and demonstrate how to handle these challenges.

#### Downloading text dataset from HF hub

In [23]:
file_path = data_dir / "financial-news-multisource" / "sample_train.json"

In [24]:

import shutil

if not file_path.exists():
    if not file_path.parent.exists():
        file_path.parent.mkdir(parents=True, exist_ok=True)

# Run the curl command
# This dataset is from Hugging Face Datasets library
# This requires an authentication token from Hugging Face
!curl -X GET \
     -H "Authorization: Bearer $HF_TOKEN" \
     -o sample_train.json \
     "https://datasets-server.huggingface.co/rows?dataset=Brianferrell787%2Ffinancial-news-multisource&config=data&split=train&offset=0&length=100"

# Move the downloaded file to the desired location
shutil.move("sample_train.json", file_path)

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  314k  100  314k    0     0   247k      0  0:00:01  0:00:01 --:--:--  247k


PosixPath('/Users/akshay/Work/data/financial-news-multisource/sample_train.json')

#### Exploring the downloaded data

In [25]:
import json
json_data = json.load(open(file_path))

In [26]:
json_data.keys()

dict_keys(['features', 'rows', 'num_rows_total', 'num_rows_per_page', 'partial'])

In [27]:
json_data["rows"][0], len(json_data["rows"])

({'row_idx': 0,
  'row': {'date': '2016-01-01T00:00:00Z',
   'text': 'New Ted Cruz Super PAC with $4M ad buy\n\n(CNN)A new super PAC supporting Ted Cruz is beginning a $4 million television advertising campaign on his behalf, the most ambitious effort yet to back him independently in the 2016 race. Stand for Truth, a group formed only six weeks ago, said Friday that it would begin $2 million TV campaigns in both Iowa and South Carolina in the coming weeks. The little-known group, which has not yet disclosed its donors, is independent of the umbrella network of super PACs, Keep the Promise, that have been blessed by the campaign. And Keep the Promise is largely unaware of the contributors and strategists behind the group, which lacks an immediate tie to the network of Cruz associates who have plotted the independent efforts to back him for over a year. Stand for Truth, however, does not appear to be a renegade organization: Keep the Promise was aware of Stand for Truth\'s advertising pl

In [28]:
for i in range(5):
    print(json_data["rows"][i]['row']['text'])
    print("-----"*20)

New Ted Cruz Super PAC with $4M ad buy

(CNN)A new super PAC supporting Ted Cruz is beginning a $4 million television advertising campaign on his behalf, the most ambitious effort yet to back him independently in the 2016 race. Stand for Truth, a group formed only six weeks ago, said Friday that it would begin $2 million TV campaigns in both Iowa and South Carolina in the coming weeks. The little-known group, which has not yet disclosed its donors, is independent of the umbrella network of super PACs, Keep the Promise, that have been blessed by the campaign. And Keep the Promise is largely unaware of the contributors and strategists behind the group, which lacks an immediate tie to the network of Cruz associates who have plotted the independent efforts to back him for over a year. Stand for Truth, however, does not appear to be a renegade organization: Keep the Promise was aware of Stand for Truth's advertising plans, even if the group is surfacing fairly late in the election calendar.

#### Defining custom dataset for text data

In [29]:
import tiktoken
## This provides byte pair encoding tokenizer used by OpenAI's GPT models
## BPE tokenizes text by breaking down rare or complex words into smaller subwords, 
## which reduces the vocabulary size and allows the model to handle rare words more effectively.
## https://github.com/openai/openai-cookbook/blob/main/examples/How_to_count_tokens_with_tiktoken.ipynb
tokenizer = tiktoken.get_encoding("cl100k_base")

sample_sentence = json_data["rows"][0]['row']['text']
tokens = tokenizer.encode(sample_sentence)
print(tokens)

[3648, 23989, 21510, 7445, 40964, 449, 400, 19, 44, 1008, 3780, 271, 3100, 9944, 8, 32, 502, 2307, 40964, 12899, 23989, 21510, 374, 7314, 264, 400, 19, 3610, 12707, 13172, 4901, 389, 813, 17981, 11, 279, 1455, 32855, 5149, 3686, 311, 1203, 1461, 29235, 304, 279, 220, 679, 21, 7102, 13, 15948, 369, 30198, 11, 264, 1912, 14454, 1193, 4848, 5672, 4227, 11, 1071, 6740, 430, 433, 1053, 3240, 400, 17, 3610, 6007, 21343, 304, 2225, 21357, 323, 4987, 13030, 304, 279, 5108, 5672, 13, 578, 2697, 22015, 1912, 11, 902, 706, 539, 3686, 36489, 1202, 33149, 11, 374, 9678, 315, 279, 48998, 4009, 315, 2307, 40964, 82, 11, 13969, 279, 7451, 11, 430, 617, 1027, 33944, 555, 279, 4901, 13, 1628, 13969, 279, 7451, 374, 14090, 41747, 315, 279, 20965, 323, 5388, 1705, 4920, 279, 1912, 11, 902, 37856, 459, 14247, 18623, 311, 279, 4009, 315, 21510, 40531, 889, 617, 68683, 279, 9678, 9045, 311, 1203, 1461, 369, 927, 264, 1060, 13, 15948, 369, 30198, 11, 4869, 11, 1587, 539, 5101, 311, 387, 264, 5790, 96337, 7471

In [30]:
tokenizer._special_tokens

{'<|endoftext|>': 100257,
 '<|fim_prefix|>': 100258,
 '<|fim_middle|>': 100259,
 '<|fim_suffix|>': 100260,
 '<|endofprompt|>': 100276}

In [31]:
tokenizer.eot_token

100257

##### Defining custom LLM Dataset class

In [32]:
class customLLMDataset(Dataset):
    """
    Custom dataset for LLM text data.
    
    This dataset loads text from JSON files, tokenizes it, and returns tensors.
    Note: Since we return tensors directly, the default DataLoader collate function
    will work correctly if all sequences have the same length. If sequences have
    variable lengths, you can either:
    1. Pad sequences in the preprocess method (uncomment the padding code below)
    2. Use a custom collate_fn in the DataLoader (see example later)
    """
    def __init__(self, file_name, maxlen=512):
        with open(file_name) as infile:
            dataset = json.load(infile)
        
        self.maxlen = maxlen
        self.data = self.preprocess(dataset)
        
    def preprocess(self, data):
        preprocessed = []
        for txt in data["rows"]:
            text = txt['row']['text']
            tokens = tokenizer.encode(text)
            tokens.append(tokenizer.eot_token)

            # Manual Padding the sentence with maximum length
            # Can be added here or in collate function
            # If padding is done here, all sequences will have the same length
            # and the default DataLoader collate function will work correctly
            # if len(tokens) <= self.maxlen:
            #     tokens += [tokenizer.eot_token] * (self.maxlen - len(tokens))
            # elif len(tokens) > self.maxlen:
            #     tokens = tokens[:self.maxlen]

            preprocessed += [tokens]
        return preprocessed
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        tokens = self.data[idx]
        
        # Return as tensor so DataLoader can batch them automatically
        return torch.tensor(tokens, dtype=torch.long)

In [33]:
train_dataset = customLLMDataset(file_path)

In [34]:
len(train_dataset)

100

In [35]:
print(train_dataset[25])

tensor([    35,  46164,   1120,  11887,    502,   3241,    311,   3009,   1202,
         38332,    505,  16706,    304,  22486,  88282,    271,   1966,   7950,
            11,  27811,  14290,  22102,     40,  11887,    264,  13746,   2373,
           315,   1202,    502,    330,    713,   1073,  11627,      1,   1887,
            11,   2555,    279,   2883,   2795,    690,   4194,  13397,   1202,
         38332,    505,  16706,   1139,  22486,  88282,     13,    578,    502,
          4668,    374,   2663,   4194,   9688,    437,  33514,  11847,   8267,
           320,     38,   6903,    705,    323,    433,    690,   1095,   3932,
          1440,    922,   5789,   1405,  27811,  11213,    374,  22486,     11,
          3060,   4245,    311,  14640,    477,   1606,    315,   7296,   4819,
            13,   1102,    596,  22102,     40,    596,   1648,    315,   2708,
          4522,    279,  67036,     11,    902,    706,   2663,    369,    810,
         19812,    315,  27811,  11213, 

### Iterating over batches using dataloader

When using the default DataLoader with variable-length sequences, PyTorch's default collate function will attempt to stack tensors. If all sequences have the same length (after padding in the dataset), this works seamlessly. If sequences have different lengths, you'll need a custom collate function.

In this example, we demonstrate both approaches:
1. Using the default collate function (works when sequences are pre-padded)
2. Using a custom collate function (handles variable-length sequences dynamically)

In [36]:
dataloader = DataLoader(train_dataset, batch_size=5)
batch = next(iter(dataloader))

RuntimeError: stack expects each tensor to be equal size, but got [576] at entry 0 and [505] at entry 1

In [ ]:
print(batch.shape)

In [ ]:
batch

When working with text data, sequences often have different lengths. There are two approaches to handle this:

1. **Padding in the Dataset class**: Uncomment the padding code in the `preprocess` method of `customLLMDataset`. This ensures all sequences have the same length before batching, allowing the default DataLoader collate function to work.

2. **Padding in a custom collate function**: Use a custom `collate_fn` to pad sequences dynamically during batching. This is useful when you want to pad only to the maximum length within each batch (more memory efficient) or when you want more control over the padding process.

**About `collate_fn`:**

The `collate_fn` is a callable/function that processes the batch before returning it from the DataLoader. The `batch` argument is a list containing all samples from the dataset. It's commonly used for:
- Padding sequential data to a consistent length
- Handling variable-sized samples
- Custom batching logic

### Custom collate for variable-length text

This custom collate function handles variable-length sequences by:
- Padding each sequence to `MAX_LEN` using the end-of-text token
- Truncating sequences longer than `MAX_LEN`
- Stacking all padded sequences into a single batch tensor of shape `[batch_size, MAX_LEN]`


In [38]:
MAX_LEN = 512
def mycollator(batch):
    """
    Custom collate function for variable-length text sequences.
    
    Args:
        batch: List of tensors, where each tensor is a tokenized sequence
        
    Returns:
        Stacked tensor of shape [batch_size, MAX_LEN] with all sequences padded/truncated
    """
    padded_batch = []
    for tokens in batch:
        # Manual Padding the sentence with maximum length
        tokens = tokens.tolist()  # Convert tensor to list for easier manipulation
        if len(tokens) <= MAX_LEN:
            tokens += [tokenizer.eot_token] * (MAX_LEN - len(tokens))
        elif len(tokens) > MAX_LEN:
            tokens = tokens[:MAX_LEN]
        padded_batch.append(torch.tensor(tokens))
    
    # Stack the padded tokens into a tensor of shape (batch_size, max_len)
    padded_batch = torch.stack(padded_batch)
    return padded_batch

**Using the custom collate function:**

When using a custom `collate_fn`, the DataLoader will:
- Pass each batch (list of samples) to the collate function
- Return the processed batch as specified by the collate function
- In this case, return a single tensor of shape `[batch_size, MAX_LEN]` with all sequences padded

This approach provides flexibility to handle variable-length sequences dynamically during batching, which can be more memory-efficient than padding all sequences in the dataset preprocessing step.


In [39]:
dataloader = DataLoader(train_dataset, batch_size=5, collate_fn=mycollator)
batch = next(iter(dataloader))

In [40]:
batch

tensor([[  3648,  23989,  21510,  ...,   3544,   9341,    304],
        [  8144,    459,   9071,  ..., 100257, 100257, 100257],
        [ 68786,   3585,    311,  ..., 100257, 100257, 100257],
        [ 56351,   3135,   4445,  ...,    520,   2204,   3115],
        [    34,  63423,   2307,  ..., 100257, 100257, 100257]])

In [41]:
print(batch.shape)

torch.Size([5, 512])
